In [ ]:
import lightning as L
import timm
import torch
from pytorch_metric_learning import miners

In [1]:
class CarsEmbedder(L.LightningModule):
    def __init__(self, margin=1.0, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = timm.create_model(model_name='efficientnet_b3', pretrained=True, num_classes=0)

        self.loss = torch.nn.TripletMarginLoss(margin=margin)
        self.miner = miners.TripletMarginMiner(margin=margin, type_of_triplets="semihard")

    def forward(self, imgs):
        return self.model.forward(imgs)

    def training_step(self, batch):
        imgs, labels = batch

        embeds = self.forward(imgs)
        anchors_indices, positives_indices, negatives_indices = self.miner(embeds, labels)

        loss = self.loss(embeds[anchors_indices], embeds[positives_indices], embeds[negatives_indices])
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

IndentationError: expected an indented block after function definition on line 22 (2323487646.py, line 25)